# Systematics: Flux, G4, MCStat, Cosmics

In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

# local imports
import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.constants import *
from analysis_village.numucc_1p0pi.files_config import *
from pyanalib.covariance import *
from makedf.mcstat import get_MCstat_unc

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = False
save_fig = save_result

today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics-other-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

In [ ]:
concat_dfs = load_df(option="systs")
mc_hdr_df = concat_dfs['hdr']
mc_evt_df = concat_dfs['evt']

In [ ]:
print("==== breakdown of selected events ====")
print(mc_evt_df.topo_categ.value_counts())
print(mc_evt_df.genie_categ.value_counts())

In [ ]:
var_config = VariableConfig.muon_momentum()

# MC Stat

In [ ]:
syst_name = "MCstat"

univ_events, cv_events = get_univ_rates(evtdf=mc_evt_df,
                                        var_config=var_config,
                                        n_univ=1000,
                                        bkgd_subtract=True,
                                        syst_name=syst_name)
ret = get_covariance_matrix(univ_events, cv_events)

In [ ]:
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
plot_frac_unc([frac_unc], var_config)

In [ ]:
matrix_type = "cov"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
             save_fig=save_fig, save_name=save_fig_name)

matrix_type = "cov_frac"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "FractionalCovariance"],
             save_fig=save_fig, save_name=save_fig_name)

# Flux

In [ ]:
syst_name = ("mc", "Flux")

univ_events, cv_events = get_univ_rates(evtdf=mc_evt_df,
                                        var_config=var_config,
                                        n_univ=100,
                                        bkgd_subtract=True,
                                        syst_name=syst_name)
ret = get_covariance_matrix(univ_events, cv_events)

In [ ]:
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
plot_frac_unc([frac_unc], var_config)

In [ ]:
matrix_type = "cov"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
             save_fig=save_fig, save_name=save_fig_name)

matrix_type = "cov_frac"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"],
             save_fig=save_fig, save_name=save_fig_name)

matrix_type = "corr"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Correlation"],
             save_fig=save_fig, save_name=save_fig_name)

# G4

In [ ]:
syst_name = ("mc", "G4")

univ_events, cv_events = get_univ_rates(evtdf=mc_evt_df,
                                        var_config=var_config,
                                        n_univ=100,
                                        bkgd_subtract=True,
                                        syst_name=syst_name)
ret = get_covariance_matrix(univ_events, cv_events)

In [ ]:
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
plot_frac_unc([frac_unc], var_config)

In [ ]:
matrix_type = "cov"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
             save_fig=save_fig, save_name=save_fig_name)

# Cosmics

take the difference between the offbeam data and intime MC as a unisim uncertainty

In [ ]:
dfs = load_df(option="cosmics_systs")
data_evt_df = dfs["data"]
mc_evt_df = dfs["mc"]


In [ ]:
# compare data and MC
fig, ax = plt.subplots()

data_events, _, _ = plt.hist(data_evt_df[var_config.var_evt_reco_col], bins=var_config.bins, histtype="step", color="red", label="Data")
mc_events, _, _   = plt.hist(mc_evt_df[var_config.var_evt_reco_col], weights=mc_evt_df["pot_scale"], bins=var_config.bins, histtype="step", color="black", label="MC")

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Events / Bin")
plt.legend()
plt.show();

In [ ]:
# treat data as a unisim systematic universe
# the MC-data difference is the variation in event rate

syst_name = "cosmics"
univ_events =np.array([cv_events + (data_events - mc_events)])

ret = get_covariance_matrix(univ_events, cv_events)

In [ ]:
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
plot_frac_unc([frac_unc], var_config)

In [ ]:
matrix_type = "cov"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
             save_fig=save_fig, save_name=save_fig_name)

# GENIE

In [ ]:
syst_name = ("mc", "GENIE")

In [ ]:
concat_dfs = load_df(option="genie_systs")
mc_hdr_df = concat_dfs['hdr']
mc_nu_df = concat_dfs['mcnu']
mc_evt_df = concat_dfs['evt']

In [ ]:
# mc_nu_df.columns = pd.MultiIndex.from_tuples([tuple(["mc"] + list(c)) for c in mc_nu_df.columns])     # match # of column levels

# # print("==== breakdown of selected events ====")
# mc_evt_df["topo_categ"] = get_topo_category(mc_evt_df)
# mc_evt_df["genie_categ"] = get_genie_category(mc_evt_df)
# print(mc_evt_df.topo_categ.value_counts())
# print(mc_evt_df.genie_categ.value_counts())

# # print("==== breakdown of all MC events ====")
# mc_nu_df["topo_categ"] = get_topo_category(mc_nu_df)
# mc_nu_df["genie_categ"] = get_genie_category(mc_nu_df)
# print(mc_nu_df.topo_categ.value_counts())
# print(mc_nu_df.genie_categ.value_counts())

In [ ]:
signal_hists(evtdf=mc_evt_df, 
             nudf=mc_nu_df, 
             var_config=var_config, 
             save_fig=save_fig, 
             save_name=None)

In [ ]:
# Note: these are background subtracted
cov_type = "xsec"
univ_events, cv_events = get_univ_rates(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name)
ret_xsec = get_covariance_matrix(univ_events, cv_events)
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

cov_type = "rate"
univ_events, cv_events = get_univ_rates(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name)
ret_rate = get_covariance_matrix(univ_events, cv_events)
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

In [ ]:
matrix_type = "cov"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name[1], matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret_xsec[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
             save_fig=save_fig, save_name=save_fig_name)
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name[1]+"_rate", matrix_type)
plot_heatmap(ret_rate[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
             save_fig=save_fig, save_name=save_fig_name)

matrix_type = "cov_frac"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name[1], matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret_xsec[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"],
             save_fig=save_fig, save_name=save_fig_name)
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name[1]+"_rate", matrix_type)
plot_heatmap(ret_rate[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"],
             save_fig=save_fig, save_name=save_fig_name)

matrix_type = "corr"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name[1], matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret_xsec[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Correlation"],
             save_fig=save_fig, save_name=save_fig_name)
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name[1]+"_rate", matrix_type)
plot_heatmap(ret_rate[matrix_type], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Correlation"],
             save_fig=save_fig, save_name=save_fig_name)

In [ ]:
frac_unc_list = [np.sqrt(np.diag(ret_xsec["cov_frac"])), np.sqrt(np.diag(ret_rate["cov_frac"]))]
plot_frac_unc(frac_unc_list, var_config)